# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset ([doi:10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which describes multiple record sets and fields derived from mass spectrometry analysis of extracellular vesicles from human mast cells.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata

print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}")
print(f"Cite as: {md.citeAs}")
print("\nKeywords:", getattr(md, 'keywords', None))

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` as per the Croissant schema.

In [ ]:
# List all record set IDs and their field IDs
print("Available record sets (by @id):")
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# For the main data record set, print field information (by @id)
if record_sets:
    main_record_set_id = record_sets[0]['@id']  # Using first as typical main table
    print(f"\nFields for record set {main_record_set_id}:")
    for field in dataset.fields(record_set=main_record_set_id):
        fid = field['@id']
        fname = field.get('name', '')
        fdesc = field.get('description', '')
        fdt = field.get('dataType', '')
        print(f"- {fid}: {fname} | Data type: {fdt} | {fdesc}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the `@id` values from the overview above.

In [ ]:
# Extract data from the principal record set
main_record_set_id = record_sets[0]['@id'] if record_sets else None
dataframes = {}

if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Columns in {main_record_set_id} record set:")
    print(df.columns.tolist())
    df.head()
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records, normalizing numeric fields, and grouping results. All field selections refer to columns by their field `@id` from the schema.

In [ ]:
# Select a numeric field for EDA
df = dataframes.get(main_record_set_id, pd.DataFrame())

# Find likely numeric fields by Croissant field `dataType`
numeric_fields = [field['@id'] for field in dataset.fields(record_set=main_record_set_id)
                  if field.get('dataType', '').lower() in ['float', 'number', 'integer']]
print('Numeric fields:', numeric_fields)

# For demonstration, pick the first numeric field
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    numeric_field = None
# Find likely categorical/grouping field
group_fields = [field['@id'] for field in dataset.fields(record_set=main_record_set_id)
                if field.get('dataType', '').lower() in ['string', 'text', 'category']]
group_field = group_fields[0] if group_fields else None
print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

if not df.empty and numeric_field in df.columns:
    # Make numeric if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()  # Use mean as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df[[numeric_field]].head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field}, showing average {numeric_field} per group:")
        print(grouped_df.head())
else:
    print("DataFrame empty or required fields not found.")

## 5. Visualization
Visualize data distributions or relationships between fields from the main record set using available numeric/categorical columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if not df.empty and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 mass spectrometry dataset using Croissant metadata and the `mlcroissant` library. We loaded dataset records, listed available fields and IDs, performed simple filtering and normalization, grouped by categorical attributes, and visualized numeric distributions.

This Croissant-based dataset provides a reproducible, machine-actionable standard for FAIR data analysis. For further work, you can refine the field selection and extend to additional statistical or machine learning workflows.